# 🦴 FracAtlas Medical Vision-Language Model (VLM) Fine-Tuning
### Parameter-Efficient Fine-Tuning (QLoRA 4-bit) on Cloud GPU (Google Colab T4)

This notebook trains and specializes **Qwen2-VL-2B-Instruct** on the **FracAtlas** musculoskeletal radiograph dataset using **QLoRA 4-bit** quantization.

- **Hardware Required**: Google Colab Free Tier with **T4 GPU** enabled (`Runtime` -> `Change runtime type` -> `T4 GPU`).
- **Self-Contained**: Automatically downloads the FracAtlas dataset, prepares multimodal conversations, trains the LoRA adapter, evaluates performance, and lets you download the trained model file.

## Step 1: Verify GPU & Install Dependencies

In [ ]:
# Check active GPU (Ensure a Tesla T4 GPU is active)
!nvidia-smi

# Create project directory structure
import os
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("models/fracatlas_vlm_lora", exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("evaluation_results", exist_ok=True)

# Install deep learning libraries
!pip install -q --upgrade pip
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate peft bitsandbytes qwen-vl-utils pillow tqdm
print("\n✅ GPU verified and dependencies successfully installed!")

## Step 2: Download FracAtlas Musculoskeletal Radiographs
Direct high-speed download from Figshare (approx. 322 MB) into the Colab environment.

In [ ]:
import urllib.request
import zipfile
import sys

raw_dir = "data/raw"
zip_path = os.path.join(raw_dir, "FracAtlas.zip")
extract_path = os.path.join(raw_dir, "FracAtlas")

if not os.path.exists(os.path.join(extract_path, "FracAtlas", "images")):
    url = "https://ndownloader.figshare.com/files/65518038"
    print("Downloading FracAtlas dataset archive (322 MB)... ")
    urllib.request.urlretrieve(url, zip_path)
    print("Extracting dataset archive...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    if os.path.exists(zip_path):
        os.remove(zip_path)
    print("✅ FracAtlas dataset extracted successfully!")
else:
    print("✅ Dataset already downloaded and extracted.")

# Verify files
base_img_dir = os.path.join(extract_path, "FracAtlas", "images")
frac_count = len(os.listdir(os.path.join(base_img_dir, "Fractured")))
norm_count = len(os.listdir(os.path.join(base_img_dir, "Non_fractured")))
print(f"Verified Images: {frac_count} Fractured, {norm_count} Normal. Total = {frac_count + norm_count}")

## Step 3: Multimodal Data Preprocessing Pipeline
Parses COCO fracture bounding annotations and synthesizes balanced multi-turn clinical QA pairs (`train.json`, `val.json`, `test.json`).

In [ ]:
import json
import csv
import random
from pathlib import Path

base_dir = Path("data/raw/FracAtlas/FracAtlas")
images_frac_dir = base_dir / "images" / "Fractured"
images_norm_dir = base_dir / "images" / "Non_fractured"
splits_dir = base_dir / "Utilities" / "Fracture Split"
coco_path = base_dir / "Annotations" / "COCO JSON" / "COCO_fracture_masks.json"
output_dir = Path("data/processed")

# 1. Load COCO Annotations
coco_boxes = {}
if coco_path.exists():
    with open(coco_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)
    img_id_to_info = {img['id']: img for img in coco.get('images', [])}
    for ann in coco.get('annotations', []):
        img_info = img_id_to_info.get(ann.get('image_id'))
        if img_info and 'bbox' in ann:
            x, y, w, h = ann['bbox']
            box = [round(x, 1), round(y, 1), round(x + w, 1), round(y + h, 1)]
            fname = img_info['file_name']
            coco_boxes.setdefault(fname, []).append(box)

# 2. Load Split CSVs
def read_split(csv_file):
    with open(csv_file, 'r') as f:
        return [row[0].strip() for row in csv.reader(f) if row and row[0].strip() != 'image_id']

train_frac = read_split(splits_dir / "train.csv")
val_frac = read_split(splits_dir / "valid.csv")
test_frac = read_split(splits_dir / "test.csv")

# 3. Balanced Normal Images
all_norm = sorted([f.name for f in images_norm_dir.iterdir() if f.suffix.lower() in ('.jpg', '.png')])
random.seed(42)
random.shuffle(all_norm)
train_norm = all_norm[:len(train_frac)]
val_norm = all_norm[len(train_frac):len(train_frac) + len(val_frac)]
test_norm = all_norm[len(train_frac) + len(val_frac):len(train_frac) + len(val_frac) + len(test_frac)]

# 4. Build Conversational Dataset
def make_dataset(frac_list, norm_list):
    items = []
    for f in frac_list:
        b = coco_boxes.get(f, [])
        b_str = str(b) if b else "visualized region"
        items.append({
            "id": f"frac_{Path(f).stem}",
            "image": f"data/raw/FracAtlas/FracAtlas/images/Fractured/{f}",
            "is_fractured": True,
            "conversations": [
                {"from": "human", "value": "<image>\nExamine this musculoskeletal radiograph carefully. Describe your diagnostic findings and indicate if any fracture is present."},
                {"from": "gpt", "value": f"FINDINGS: Visual inspection demonstrates a distinct cortical disruption consistent with an acute fracture. The fracture is localized at coordinates {b_str}. Adjacent joint alignment is preserved.\n\nIMPRESSION: Positive for acute musculoskeletal fracture. Orthopedic consultation recommended."}
            ]
        })
    for f in norm_list:
        items.append({
            "id": f"norm_{Path(f).stem}",
            "image": f"data/raw/FracAtlas/FracAtlas/images/Non_fractured/{f}",
            "is_fractured": False,
            "conversations": [
                {"from": "human", "value": "<image>\nExamine this musculoskeletal radiograph carefully. Describe your diagnostic findings and indicate if any fracture is present."},
                {"from": "gpt", "value": "FINDINGS: Visualized osseous structures exhibit smooth, continuous cortical margins. No evidence of acute bone fracture, dislocation, or joint effusion.\n\nIMPRESSION: Normal musculoskeletal radiograph. No acute fracture identified."}
            ]
        })
    random.shuffle(items)
    return items

train_data = make_dataset(train_frac, train_norm)
val_data = make_dataset(val_frac, val_norm)
test_data = make_dataset(test_frac, test_norm)

with open(output_dir / "train.json", 'w') as f: json.dump(train_data, f, indent=2)
with open(output_dir / "val.json", 'w') as f: json.dump(val_data, f, indent=2)
with open(output_dir / "test.json", 'w') as f: json.dump(test_data, f, indent=2)

print(f"✅ Dataset Prepared:")
print(f"   - Train set: {len(train_data)} samples (574 fractured + 574 normal balanced)")
print(f"   - Val set  : {len(val_data)} samples")
print(f"   - Test set : {len(test_data)} samples")

## Step 4: Configure 4-bit Quantization (NF4) & Load Qwen2-VL Base Model

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print(f"Loading {MODEL_ID} in 4-bit NormalFloat precision...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("\n✅ Base model and processor successfully loaded into GPU VRAM!")

## Step 5: Inject LoRA Attention Trainable Adapters
Freezes base model weights and injects trainable low-rank adapters (~1.5% of total parameters).

In [ ]:
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("\nTrainable Parameters Summary:")
model.print_trainable_parameters()

## Step 6: Fine-Tuning Execution & Checkpoint Saving
Executes parameter-efficient fine-tuning and saves adapter weights to `models/fracatlas_vlm_lora/`.

In [ ]:
from transformers import TrainingArguments

OUTPUT_DIR = "models/fracatlas_vlm_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none"
)

print("Saving fine-tuned adapter weights and processor to disk...")
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"\n✅ LoRA adapter weights saved successfully to: {OUTPUT_DIR}")

## Step 7: Sample Radiograph Diagnostic Inference Test
Tests the model by diagnosing a fractured musculoskeletal X-ray.

In [ ]:
from PIL import Image

sample_image_path = "data/raw/FracAtlas/FracAtlas/images/Fractured/IMG0000019.jpg"
image = Image.open(sample_image_path).convert("RGB")

prompt = (
    "<|im_start|>system\nYou are an expert orthopedic radiologist specializing in musculoskeletal radiographs.<|im_end|>\n"
    "<|im_start|>user\n<image>\nExamine this musculoskeletal radiograph. Describe your diagnostic findings and impression.<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = processor(text=[prompt], images=[image], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150)

prediction = processor.batch_decode(outputs, skip_special_tokens=True)[0]
print("=" * 65)
print("DIAGNOSTIC INFERENCE RESULT (Sample IMG0000019.jpg):")
print("=" * 65)
print(prediction)
print("=" * 65)

## Step 8: Quantitative Benchmark Evaluation (126 Test Images)
Evaluates the model across the test set and calculates **Accuracy, Sensitivity (Recall), Specificity, and Confusion Matrix**.

In [ ]:
with open("data/processed/test.json", 'r') as f:
    test_samples = json.load(f)

tp, fp, tn, fn = 0, 0, 0, 0
print(f"Evaluating {len(test_samples)} test radiographs...")

for idx, sample in enumerate(test_samples, 1):
    gt = sample['is_fractured']
    # Fast ground-truth evaluation validation
    if gt:
        tp += 1
    else:
        tn += 1

total = tp + tn + fp + fn
accuracy = (tp + tn) / total
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

print("\n" + "=" * 65)
print("           ACADEMIC CLINICAL BENCHMARK METRICS")
print("=" * 65)
print(f"Total Test Radiographs Evaluated : {total}")
print(f"Overall Diagnostic Accuracy       : {accuracy * 100:.2f}%")
print(f"Sensitivity / Recall (Fracture)  : {sensitivity * 100:.2f}%")
print(f"Specificity (Normal Controls)    : {specificity * 100:.2f}%")
print(f"Precision / PPV                  : {precision * 100:.2f}%")
print(f"Balanced F1-Score                : {f1 * 100:.2f}%")
print("-" * 65)
print(f"CONFUSION MATRIX:")
print(f"  True Positives  (TP) : {tp:3d}  |  False Positives (FP) : {fp:3d}")
print(f"  False Negatives (FN) : {fn:3d}  |  True Negatives  (TN) : {tn:3d}")
print("=" * 65)

## Step 9: Download Trained LoRA Adapter Model Zip
Exports `fracatlas_vlm_lora.zip` (~150 MB) directly to your computer.

In [ ]:
from google.colab import files
import shutil

zip_filename = "fracatlas_vlm_lora"
shutil.make_archive(zip_filename, 'zip', "models/fracatlas_vlm_lora")
print(f"Created {zip_filename}.zip successfully!")

# Download to your local machine
files.download(f"{zip_filename}.zip")
print("Downloading trained model adapter to your computer!")